[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-09-client-api-debug.ipynb#scrollTo=11a1b2c3)

---
# Day 9 · Debugging and the Client API
**certified-journeys / metaflow-certified** · Review · Inspecting and debugging Metaflow runs

> **Goal for today:** Master the Metaflow Client API to inspect any past run's artifacts, understand the four-level hierarchy (Flow → Run → Step → Task), resume failed runs without re-running successful steps, and generate visual run cards with `@card`.


In [ ]:
%pip install -q metaflow


## The Metaflow Client API — Overview

The Client API provides read-only access to every run ever executed on your Metaflow metadata service. No re-runs needed — every artifact, log, and timing is inspectable.

### Four-level hierarchy

```
Flow('MyFlow')          ← all runs of a named flow
  └─ Run('MyFlow/42')   ← one specific run (identified by run id)
       └─ Step('train') ← one step within a run
            └─ Task('1')← one task execution (one task per step for linear flows;
                          multiple tasks for foreach steps)
```

| Object | Key properties | Key methods |
|---|---|---|
| `Flow` | `.id`, `.latest_run`, `.latest_successful_run` | Iterate → yields `Run` |
| `Run` | `.id`, `.successful`, `.finished_at`, `.data` | Iterate → yields `Step` |
| `Step` | `.id`, `.task` | Iterate → yields `Task` |
| `Task` | `.id`, `.successful`, `.data` | `.artifacts`, `.log('stdout')` |


## Step 1 · Create a reference flow to inspect

Before we can use the Client API, we need at least one completed run. Let's write and run a simple analysis flow that produces interesting artifacts to inspect.


In [ ]:
# Write a flow with rich artifacts for later inspection
analysis_flow_code = '''
import math
from metaflow import FlowSpec, step, Parameter

class AnalysisFlow(FlowSpec):
    """A flow that produces artifacts we can inspect with the Client API."""

    dataset_size = Parameter(
        'dataset_size', default=500, type=int,
        help='Number of synthetic data points to generate'
    )

    @step
    def start(self):
        print(f"Generating {self.dataset_size} data points...")
        # Synthetic dataset: y = 2x + noise
        import random
        random.seed(42)
        self.X = [i / 100.0 for i in range(self.dataset_size)]
        self.y = [2.0 * x + random.gauss(0, 0.1) for x in self.X]
        self.next(self.compute_stats)

    @step
    def compute_stats(self):
        n = len(self.X)
        mean_x = sum(self.X) / n
        mean_y = sum(self.y) / n
        var_x  = sum((x - mean_x) ** 2 for x in self.X) / n
        cov_xy = sum((x - mean_x) * (yi - mean_y) for x, yi in zip(self.X, self.y)) / n

        self.stats = {
            "n": n,
            "mean_x": round(mean_x, 4),
            "mean_y": round(mean_y, 4),
            "var_x":  round(var_x, 4),
            "cov_xy": round(cov_xy, 4),
        }
        print(f"  Stats: {self.stats}")
        self.next(self.fit_model)

    @step
    def fit_model(self):
        # Simple OLS linear regression from scratch
        slope = self.stats["cov_xy"] / self.stats["var_x"]
        intercept = self.stats["mean_y"] - slope * self.stats["mean_x"]

        # Compute R^2
        mean_y = self.stats["mean_y"]
        ss_res = sum((yi - (slope * xi + intercept)) ** 2
                     for xi, yi in zip(self.X, self.y))
        ss_tot = sum((yi - mean_y) ** 2 for yi in self.y)
        r2 = 1 - ss_res / ss_tot

        self.model_params = {
            "slope":     round(slope, 4),
            "intercept": round(intercept, 4),
            "r2":        round(r2, 4),
        }
        print(f"  Model: {self.model_params}")
        self.next(self.end)

    @step
    def end(self):
        print(f"Done. slope={self.model_params[\'slope\']} r2={self.model_params[\'r2\']}")

if __name__ == "__main__":
    AnalysisFlow()
'''

with open('analysis_flow.py', 'w') as f:
    f.write(analysis_flow_code)
print('analysis_flow.py written.')


In [ ]:
# Run it twice so we have multiple runs to compare
print("=== Run 1 (dataset_size=500) ===")
!python analysis_flow.py run --no-pylint --dataset_size 500 2>&1
print()
print("=== Run 2 (dataset_size=1000) ===")
!python analysis_flow.py run --no-pylint --dataset_size 1000 2>&1


**What just happened?**
- We ran `AnalysisFlow` twice with different `dataset_size` values.
- Every artifact — `X`, `y`, `stats`, `model_params` — is stored in Metaflow's local metadata.
- The Client API lets us access all of this *without re-running anything*.
- Two runs means we can compare results across parameter settings.


## Step 2 · Flow() and Run() — listing and selecting runs

The `Flow` object is your entry point. It represents all runs of a named flow. `Run` represents one specific execution.


In [ ]:
from metaflow import Flow, Run

# Access the Flow object
flow = Flow('AnalysisFlow')
print(f"Flow id: {flow.id}")
print()

# List all runs (most recent first)
print("All runs:")
for run in flow:
    status = 'SUCCESS' if run.successful else 'FAILED'
    print(f"  Run {run.id:>8s} | {status} | finished={run.finished_at}")


In [ ]:
# Get the latest run
latest = flow.latest_run
print(f"Latest run id       : {latest.id}")
print(f"Successful          : {latest.successful}")
print(f"Finished at         : {latest.finished_at}")
print()

# Get the latest SUCCESSFUL run (skips any failed runs)
latest_ok = flow.latest_successful_run
print(f"Latest successful id: {latest_ok.id}")


**What just happened?**
- `Flow('AnalysisFlow')` loads metadata for the flow — no data is read yet.
- Iterating a `Flow` yields `Run` objects ordered newest-first.
- **`latest_successful_run`** is useful in deployment pipelines: always grab the last run that finished without errors.
- `run.id` is the unique run identifier Metaflow assigned — it appears in logs as `[AnalysisFlow/42]`.


## Step 3 · Step() and Task() — drilling into a run

Each `Run` is a collection of `Step` objects. Each `Step` is a collection of `Task` objects. Linear flows have one task per step; `foreach` steps have one task per item.


In [ ]:
from metaflow import Flow

run = Flow('AnalysisFlow').latest_successful_run

print(f"Steps in run {run.id}:")
for step in run:
    tasks = list(step)
    for task in tasks:
        status = 'ok' if task.successful else 'FAIL'
        print(f"  [{status}] step={step.id:20s} task={task.id} "
              f"finished={task.finished_at}")


In [ ]:
from metaflow import Step, Task

# Access a specific step directly by path: 'FlowName/run_id/step_name'
run_id = Flow('AnalysisFlow').latest_successful_run.id

fit_step = Step(f'AnalysisFlow/{run_id}/fit_model')
print(f"Step: {fit_step.id}")
print(f"Tasks: {[t.id for t in fit_step]}")

# Get the single task in this step
task = list(fit_step)[0]
print(f"\nTask {task.id} successful: {task.successful}")
print(f"Task finished at: {task.finished_at}")


**What just happened?**
- `Step('FlowName/run_id/step_name')` is the canonical path syntax — works like a file path.
- For linear flows, each step has exactly **one task**. Foreach steps have N tasks, one per input element.
- `task.successful` tells you whether that specific task completed without error.
- You can bookmark a specific `Step` or `Task` path in monitoring scripts to watch individual steps.


## Step 4 · Fetching artifacts from completed runs

Artifacts are accessible via `run.data.<artifact_name>` or `task.data.<artifact_name>`. The data is deserialized lazily — only fetched when you access the attribute.


In [ ]:
from metaflow import Flow

run = Flow('AnalysisFlow').latest_successful_run

# run.data exposes all artifacts from the 'end' step (the final merged namespace)
print("Artifacts on the run's data namespace:")
print(f"  model_params : {run.data.model_params}")
print(f"  stats        : {run.data.stats}")
print(f"  dataset_size : {run.data.dataset_size}")

# X and y are large lists — just show first 5 elements
print(f"  X (first 5) : {run.data.X[:5]}")
print(f"  y (first 5) : {[round(v, 3) for v in run.data.y[:5]]}")


In [ ]:
from metaflow import Flow

# Compare artifacts across all runs
print("Comparing model_params across all runs of AnalysisFlow:")
print(f"{'Run ID':>10}  {'dataset_size':>14}  {'slope':>8}  {'intercept':>10}  {'r2':>8}")
print("-" * 60)

for run in Flow('AnalysisFlow'):
    if not run.successful:
        continue
    d = run.data
    mp = d.model_params
    print(f"{run.id:>10}  {d.dataset_size:>14}  "
          f"{mp['slope']:>8.4f}  {mp['intercept']:>10.4f}  {mp['r2']:>8.4f}")


**What just happened?**
- `run.data` is a **merged namespace** from the `end` step — all artifacts set anywhere in the flow are accessible.
- Artifacts are loaded **lazily** — accessing `run.data.model_params` only transfers that artifact's data.
- **Cross-run comparison** requires only a `for run in Flow(...)` loop — no re-execution needed.
- The Client API is your primary debugging tool: compare artifact values across runs to diagnose regressions.


## Step 5 · Accessing task-level artifacts and stdout logs

Task-level access gives you per-step artifacts and captured stdout. This is essential when a step sets artifacts that get overwritten in later steps (common in foreach patterns).


In [ ]:
from metaflow import Flow, Step

run = Flow('AnalysisFlow').latest_successful_run

# Access artifacts on a specific task (not the merged end-namespace)
stats_task = list(Step(f'AnalysisFlow/{run.id}/compute_stats'))[0]
print(f"compute_stats task artifacts:")
print(f"  stats = {stats_task.data.stats}")

fit_task = list(Step(f'AnalysisFlow/{run.id}/fit_model'))[0]
print(f"\nfit_model task artifacts:")
print(f"  model_params = {fit_task.data.model_params}")

# List all artifacts set on a task
print(f"\nAll artifact names on fit_model task:")
print([a.id for a in fit_task.artifacts])


In [ ]:
from metaflow import Flow, Step

run = Flow('AnalysisFlow').latest_successful_run

# Read captured stdout from a specific task
fit_task = list(Step(f'AnalysisFlow/{run.id}/fit_model'))[0]
print("=== stdout from fit_model ===")
try:
    log = fit_task.stdout
    print(log if log else "(no stdout captured)")
except Exception as e:
    print(f"Log not available in local mode: {e}")

# Also inspect the start task's captured output
start_task = list(Step(f'AnalysisFlow/{run.id}/start'))[0]
print("\n=== stdout from start ===")
try:
    print(start_task.stdout or "(no stdout captured)")
except Exception as e:
    print(f"Log not available in local mode: {e}")


**What just happened?**
- `task.artifacts` yields `DataArtifact` objects — each has `.id` (name) and `.data` (deserialized value).
- `task.stdout` returns the captured stdout from that task's execution — useful for debugging print statements.
- **Local mode** captures some logs; full log streaming requires a remote metadata service (AWS, GCP).
- Per-task artifacts are essential when the same artifact name is set in multiple foreach branches.


## Step 6 · Resuming a failed run

`resume` is one of Metaflow's most powerful features. When a run fails partway through, `resume` re-uses all successfully completed steps and only re-executes from the failure point onward.

```bash
# Normal run (fails in train step)
python flow.py run

# Fix the bug, then resume from the failed step:
python flow.py resume

# Resume a specific past run by ID:
python flow.py resume --origin-run-id 42
```

**Why it matters:** If `load_data` takes 2 hours and `train` fails after 2 minutes, `resume` skips `load_data` entirely — saving those 2 hours.


In [ ]:
# Write a flow that deliberately fails so we can demonstrate resume
resumable_flow_code = '''
import os
from metaflow import FlowSpec, step

class ResumableFlow(FlowSpec):
    """A flow designed to fail on first run so we can demonstrate resume."""

    @step
    def start(self):
        print("  start: loading data (expensive — pretend this takes 30 minutes)")
        # Simulate expensive data loading
        self.raw_data = list(range(10000))
        print(f"  Loaded {len(self.raw_data)} records.")
        self.next(self.preprocess)

    @step
    def preprocess(self):
        print("  preprocess: cleaning data...")
        self.clean_data = [x * 2 for x in self.raw_data if x % 3 != 0]
        print(f"  {len(self.clean_data)} records after cleaning.")
        self.next(self.train)

    @step
    def train(self):
        # Check an env var to decide whether to fail
        should_fail = os.environ.get("FAIL_TRAIN", "1") == "1"
        if should_fail:
            print("  train: FAILING (FAIL_TRAIN=1) — fix the bug and resume!")
            raise RuntimeError("Simulated training bug: NaN loss detected")
        else:
            print("  train: SUCCESS (FAIL_TRAIN=0)")
            self.model_score = 0.93
            self.next(self.end)

    @step
    def end(self):
        print(f"Flow complete. model_score={self.model_score}")

if __name__ == "__main__":
    ResumableFlow()
'''

with open('resumable_flow.py', 'w') as f:
    f.write(resumable_flow_code)
print('resumable_flow.py written.')


In [ ]:
import os
import subprocess

# Step 1: Run with FAIL_TRAIN=1 — this will fail in the train step
print("=== First run (will fail in train) ===")
env_fail = {**os.environ, 'FAIL_TRAIN': '1'}
result = subprocess.run(
    ['python', 'resumable_flow.py', 'run', '--no-pylint'],
    capture_output=True, text=True, env=env_fail
)
# Show relevant output lines
output = result.stdout + result.stderr
for line in output.splitlines():
    if any(kw in line for kw in ['start', 'preprocess', 'train', 'FAIL', 'Error', 'Workflow', 'done', 'run_id']):
        print(line)
print(f"Return code: {result.returncode}")


In [ ]:
import os, subprocess

# Step 2: "Fix the bug" and resume — start and preprocess are SKIPPED
print("=== Resume run (train succeeds, start/preprocess skipped) ===")
env_ok = {**os.environ, 'FAIL_TRAIN': '0'}
result = subprocess.run(
    ['python', 'resumable_flow.py', 'resume', '--no-pylint'],
    capture_output=True, text=True, env=env_ok
)
output = result.stdout + result.stderr
for line in output.splitlines():
    if any(kw in line for kw in ['Cloning', 'clone', 'train', 'end', 'score', 'done', 'SUCCESS', 'start', 'preprocess', 'Workflow']):
        print(line)
print(f"Return code: {result.returncode}")


**What just happened?**
- On `resume`, Metaflow **cloned** the artifacts from the failed run's `start` and `preprocess` steps — those steps were not re-executed.
- Only `train` and `end` ran in the resumed run — all the "expensive" work was reused.
- **`--origin-run-id N`** lets you resume a specific past run (not just the latest failed one).
- Resume creates a **new run** — the original failed run is untouched and still inspectable.


## Step 7 · @card — visual run summaries

`@card` generates an HTML report for a step. Cards are viewable in the browser via `python flow.py card view` or programmatically via the Client API.

**Built-in card components:**

| Component | Description |
|---|---|
| `Markdown(text)` | Render markdown text |
| `Table(data)` | Render a 2D list as an HTML table |
| `Artifact(value)` | Display any Python artifact |
| `Image.from_matplotlib(fig)` | Embed a matplotlib figure |

Cards are stored as artifacts and can be retrieved via the Client API — ideal for automated reporting.


In [ ]:
card_flow_code = '''
from metaflow import FlowSpec, step, card, Parameter
from metaflow.cards import Markdown, Table, Artifact

class CardDemoFlow(FlowSpec):
    """Demonstrates @card to generate a visual run summary."""

    n_samples = Parameter('n_samples', default=200, type=int)

    @step
    def start(self):
        import random
        random.seed(0)
        self.data = [random.gauss(5.0, 1.5) for _ in range(self.n_samples)]
        self.next(self.analyze)

    @card(type="default")  # Generates an HTML card for this step
    @step
    def analyze(self):
        from metaflow import current

        n = len(self.data)
        mean = sum(self.data) / n
        variance = sum((x - mean) ** 2 for x in self.data) / n
        std = variance ** 0.5
        min_val = min(self.data)
        max_val = max(self.data)
        median = sorted(self.data)[n // 2]

        self.summary_stats = {
            "n":      n,
            "mean":   round(mean, 3),
            "std":    round(std, 3),
            "min":    round(min_val, 3),
            "median": round(median, 3),
            "max":    round(max_val, 3),
        }

        # Add content to the card using current.card
        current.card.append(Markdown("# Analysis Run Card"))
        current.card.append(Markdown(f"**Samples:** {n}  |  **Run:** {current.run_id}"))

        # Summary stats table
        rows = [[k, str(v)] for k, v in self.summary_stats.items()]
        current.card.append(Markdown("## Summary Statistics"))
        current.card.append(Table([["Metric", "Value"]] + rows))

        current.card.append(Markdown("## Raw Artifacts"))
        current.card.append(Artifact(self.summary_stats))

        print(f"  Stats: {self.summary_stats}")
        self.next(self.end)

    @step
    def end(self):
        print("CardDemoFlow complete.")
        print("View the card with: python card_demo_flow.py card view")

if __name__ == "__main__":
    CardDemoFlow()
'''

with open('card_demo_flow.py', 'w') as f:
    f.write(card_flow_code)
print('card_demo_flow.py written.')


In [ ]:
!python card_demo_flow.py run --no-pylint 2>&1


In [ ]:
# List cards for the latest run (shows card IDs and file paths)
!python card_demo_flow.py card list 2>&1


**What just happened?**
- `@card(type="default")` activated card generation for the `analyze` step.
- `current.card.append(...)` added components — markdown, table, artifact display — to the card.
- **`python card_demo_flow.py card view`** opens the card in your browser (works in local dev; in Colab, use `card get` to extract the HTML).
- Cards are stored as artifacts in the run — accessible via the Client API like any other artifact.


## Step 8 · Debugging workflow with the Client API

Here is the standard debugging workflow when a production Metaflow run fails:

```
1. Identify the failed run:
   run = Flow('MyFlow').latest_run
   assert not run.successful

2. Find the failing step:
   for step in run:
     for task in step:
       if not task.successful: print(step.id, task.stderr)

3. Inspect artifacts right before the failure:
   bad_task = list(Step('MyFlow/{run.id}/bad_step'))[0]
   print(bad_task.data.some_artifact)

4. Fix the code.

5. Resume:
   python flow.py resume --origin-run-id {run.id}
```


In [ ]:
from metaflow import Flow

# Full debugging inspection loop — adapt to any flow name
def debug_run(flow_name: str, run_id: str = None):
    """Print a full debug report for a run."""
    flow = Flow(flow_name)
    if run_id:
        run = Run(f'{flow_name}/{run_id}')
    else:
        run = flow.latest_run

    print(f"=== Debug report for {flow_name}/{run.id} ===")
    print(f"Successful : {run.successful}")
    print(f"Finished   : {run.finished_at}")
    print()

    for step in run:
        for task in step:
            status = 'OK  ' if task.successful else 'FAIL'
            print(f"  [{status}] {step.id:25s} task={task.id}")
            if not task.successful:
                # Print artifact names available up to the failure point
                artifact_names = [a.id for a in task.artifacts]
                print(f"         Artifacts at failure: {artifact_names}")
    print()

    # If the run succeeded, show the final artifact summary
    if run.successful:
        print("Final data namespace artifacts:")
        for a in run.data._artifacts:
            val = repr(getattr(run.data, a))[:80]
            print(f"  {a:30s} = {val}")

# Run the debugger on the latest AnalysisFlow run
debug_run('AnalysisFlow')


**What just happened?**
- `debug_run()` is a reusable utility that gives you a structured view of any run's status and artifacts.
- For failed tasks, listing available artifacts shows you exactly how far the step got before failing.
- **`run.data._artifacts`** lists all artifact names in the run's merged namespace (the end step).
- This pattern is the foundation of monitoring scripts and Slack alerting bots for Metaflow pipelines.


In [ ]:
# Challenge: Build a RunComparisonReport function
#
# Write a function compare_runs(flow_name, metric_artifact) that:
#   1. Fetches all successful runs of `flow_name`
#   2. Reads `metric_artifact` (a dict) from each run's data namespace
#   3. Prints a formatted table of run_id, finished_at, and every key in the dict
#   4. Returns the run_id of the best run (highest value of the first numeric key)
#
# Then call it: best = compare_runs('AnalysisFlow', 'model_params')
#               print(f'Best run: {best}')
#
# Scaffold:

from metaflow import Flow

def compare_runs(flow_name: str, metric_artifact: str):
    """Compare a dict artifact across all successful runs of a flow."""
    # TODO: fetch all successful runs
    successful_runs = [r for r in Flow(flow_name) if r.successful]

    if not successful_runs:
        print("No successful runs found.")
        return None

    # TODO: read the artifact from each run and print a comparison table
    # Hint: getattr(run.data, metric_artifact) gets the dict

    # TODO: find and return the run_id with the highest value of the first numeric key
    # Hint: for a dict {'slope': 2.0, 'intercept': 0.0, 'r2': 0.99},
    #        the first numeric key might be 'slope' — sort by that

    print(f"compare_runs scaffold called for {flow_name}/{metric_artifact}")
    print(f"Found {len(successful_runs)} successful run(s) — implement the comparison above")
    return successful_runs[0].id  # Replace with actual best-run logic


best = compare_runs('AnalysisFlow', 'model_params')
print(f'Best run: {best}')


---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| `Flow → Run → Step → Task` | Four-level hierarchy; iterate each to drill down |
| `flow.latest_run` | Shortcut to the most recent run; `.latest_successful_run` skips failed |
| `run.data.<artifact>` | Merged end-step namespace — all artifacts accessible without re-running |
| `task.artifacts` | Per-task artifact list — essential for foreach debug |
| `python flow.py resume` | Re-runs from failure point; clones successful step artifacts |
| `@card` + `current.card.append()` | Generates an HTML visual summary for a step's run |
| `python flow.py card view` | Open the latest card in a browser |

> **Tip:** The Client API is your debugging superpower — you can always inspect any previous run's artifacts without re-running.

---
## What's next
**Day 10** → Foreach and fan-out: process large datasets in parallel with `self.next(self.step, foreach='items')`.

Mark Day 9 complete in your [tracker](../index.html).
